# Class 16 · Exceptions, CSV and Modules

Python · AIML Jul26 Batch 3

Every example from the class, in order. Run a cell with **Shift+Enter**. Cells that use `input()` wait for you to type an answer below them.

- Class page: https://prateekscaler.github.io/AIML-Jul26-Batch3-Python/Class-16-Exceptions-CSV-And-Modules/
- Notes, assignment and interview questions: https://github.com/prateekScaler/AIML-Jul26-Batch3-Python/tree/main/Class-16-Exceptions-CSV-And-Modules


## Try except

`01_try_except.py`

try/except catches a named error and lets the program go on.
First on errors from earlier classes, then on files.


In [ ]:
bill = 1200
people = 0
try:
    share = bill / people
    print("Each pays", share)
except ZeroDivisionError:
    print("Nobody to split with.")

marks = [72, 85, 90]
try:
    print(marks[5])
except IndexError:
    print("No such position.")

contacts = {"sumit@ybl": "Sumit"}
try:
    print(contacts["vamsi@oksbi"])
except KeyError:
    print("Not in your contacts.")

# two excepts: each error finds the one with its own name
def split(bill, people_text):
    try:
        people = int(people_text)
        return bill / people
    except ValueError:
        return "Please type a number."
    except ZeroDivisionError:
        return "Nobody to split with."

print(split(1200, "4"))
print(split(1200, "four"))
print(split(1200, "0"))

# the same plan for a missing file
try:
    file = open("missing.txt")
    print("A")                    # skipped: the error happened on the line above
except FileNotFoundError:
    print("B")
print("C")

# more than one except: Python uses the first one that matches
def read_notes(filename):
    try:
        with open(filename, "r") as file:
            return file.read()
    except FileNotFoundError:
        return "Error: file not found."
    except PermissionError:
        return "Error: not allowed to read this file."

with open("passbook.txt", "w") as f:
    f.write("Paid 600 to Sumit\nPaid 250 to Vamsi\n")

print(read_notes("passbook.txt"))
print(read_notes("secret_diary.txt"))

# not only files fail: text that is not a number breaks int()
try:
    amount = int("abc")
    print("Amount:", amount)
except ValueError:
    print("Please type a number.")

# the except is chosen by the kind of error, not by position
try:
    amount = int("12a")
    file = open("passbook.txt")
except FileNotFoundError:
    print("A")
except ValueError:
    print("B")                      # B

# a bare except hides bugs: the typo below is reported as a missing file
try:
    with open("passbook.txt") as file:
        text = fille.read()
    print(len(text))
except:
    print("No passbook yet.")       # wrong message: the file is there


## Csv files

`02_csv_files.py`

csv reads and writes comma-separated rows. Every value read back is a string.


In [ ]:
import csv

# why not split: a comma inside a value breaks it
line = '"Sumit Kumar, Jr",600'
print(len(line.split(",")))                  # 3: wrong
print(next(csv.reader([line])))              # ['Sumit Kumar, Jr', '600']

# writing: newline="" stops blank lines between rows on some systems
rows = [["payee", "amount"], ["Sumit", 600], ["Vamsi", 250]]
with open("payments.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

# reading: one list per row, header first
with open("payments.csv", "r") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

# DictReader: the header gives the keys
total = 0
with open("payments.csv", "r") as file:
    for row in csv.DictReader(file):
        print(row["payee"], row["amount"])
        total += int(row["amount"])
print("Total:", total)


## Module file: `greetings.py`

Run this cell first. It saves `greetings.py` so the next cells can import it.

An ordinary Python file. Another file imports it by name.


In [ ]:
%%writefile greetings.py
# Module file: greetings.py
# An ordinary Python file. Another file imports it by name.

def say_hello(name):
    return f"Hello, {name}!"

def say_goodbye(name):
    return f"Goodbye, {name}!"


## Import modules

`03_import_modules.py`

Four ways to import. greetings.py must be in the same folder.


In [ ]:
import greetings
print(greetings.say_hello("Parveen"))

from greetings import say_hello        # just one name, no prefix needed
print(say_hello("Prasad"))

import greetings as g                  # a short nickname for the module
print(g.say_goodbye("Prasad"))

from greetings import *                # everything: easy to clash, so avoid it
print(say_goodbye("Parveen"))


## Standard library

`04_standard_library.py`

The standard library: modules that come with Python. No install needed.


In [ ]:
import math
import random
import datetime
import os
import uuid

print(math.sqrt(49))           # 7.0
print(random.randint(1, 6))    # any number from 1 to 6
print(datetime.date.today())   # today's date
print(os.getcwd())             # the folder the program runs in
print(uuid.uuid4())            # a unique ID, different every time


## Log analyzer

`05_log_analyzer.py`

Mini project: read visits from a CSV, count the errors, write a summary.


In [ ]:
import csv

with open("visits.csv", "w", newline="") as f:
    f.write("page,status_code\n/home,200\n/login,200\n/admin,403\n/checkout,500\n")

def analyze_visits(filename):
    try:
        with open(filename, "r") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
    except FileNotFoundError:
        print(f"Error: {filename} not found.")
        return

    total = len(rows)
    errors = [r for r in rows if int(r["status_code"]) >= 400]

    print(f"Total requests: {total}")
    print(f"Errors: {len(errors)}")

    with open("summary.txt", "w") as file:
        file.write(f"Total requests: {total}\n")
        file.write(f"Errors: {len(errors)}\n")

analyze_visits("visits.csv")
analyze_visits("nothing_here.csv")

with open("summary.txt") as f:
    print(f.read())


## Module file: `paisapay_checks.py`

Run this cell first. It saves `paisapay_checks.py` so the next cells can import it.

PaisaPay's checks, moved into their own module. paisapay_v16.py imports them.


In [ ]:
%%writefile paisapay_checks.py
# Module file: paisapay_checks.py
# PaisaPay's checks, moved into their own module. paisapay_v16.py imports them.
import re


def clean(text):
    return text.strip().lower()


def is_valid_upi_id(upi_id):
    # letters, digits or dots, then @, then a bank handle of letters only
    return re.fullmatch(r"[a-z0-9.]+@[a-z]+", upi_id) is not None


def is_valid_pan(pan):
    # five letters, four digits, one letter: ABCDE1234F
    return re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", pan) is not None


def read_bank_sms(sms):
    # "Rs.600 debited from a/c XX4321 to SUMIT on 04-11-26 via UPI"
    m = re.search(r"Rs\.(\d+) debited from a/c XX(\d{4}) to (\w+)", sms)
    if m is None:
        return None
    amount, tail, payee = m.groups()
    return int(amount), tail, payee.title()


def mode_name(mode):
    match mode:
        case "upi":
            return "UPI"
        case "card":
            return "Card"
        case "netbanking":
            return "Net banking"
        case _:
            return "Unknown mode"


## Paisapay v16

`paisapay_v16.py`

PaisaPay v16: PaisaPay now remembers. The balance is saved in a text file and every
payment is added to a CSV file, so a second run starts where the first one stopped.
The checks moved into their own module, paisapay_checks.py, in the same folder.


In [ ]:
import csv
import datetime
import os
from functools import reduce
from paisapay_checks import (clean, is_valid_upi_id, is_valid_pan,
                             read_bank_sms, mode_name)

BALANCE_FILE = "paisapay_balance.txt"
HISTORY_FILE = "paisapay_history.csv"

print("Welcome to PaisaPay")

contacts = {"rajani@okaxis": "Rajani", "sumit@ybl": "Sumit"}


def name_for(upi_id):
    return contacts.get(upi_id, upi_id.split("@")[0].title())


def load_balance(default):
    try:
        with open(BALANCE_FILE, "r") as file:
            return int(file.read().strip())
    except FileNotFoundError:
        return default                       # first run: no file yet
    except ValueError:
        print(f"The balance file is damaged. Starting from ₹{default}.")
        return default


def save_balance(balance):
    with open(BALANCE_FILE, "w") as file:    # w: replace the old number
        file.write(str(balance))


def load_history():
    try:
        with open(HISTORY_FILE, "r") as file:
            rows = list(csv.DictReader(file))
    except FileNotFoundError:
        return []
    # every value comes back as a string, so the numbers need int()
    return [Transaction(r["payee"], int(r["amount"]), r["mode"],
                        int(r["fee"]), int(r["cashback"]), r["date"])
            for r in rows]


def save_payment(t):
    new_file = not os.path.exists(HISTORY_FILE)
    with open(HISTORY_FILE, "a", newline="") as file:   # a: add, never erase
        writer = csv.writer(file)
        if new_file:
            writer.writerow(["date", "payee", "amount",
                             "mode", "fee", "cashback"])
        writer.writerow([t.date, t.payee, t.amount,
                         t.mode, t.fee, t.cashback])


class Transaction:
    def __init__(self, payee, amount, mode, fee, cashback, date):
        self.payee = payee
        self.amount = amount
        self.mode = mode
        self.fee = fee
        self.cashback = cashback
        self.date = date

    def describe(self):
        return f"{self.date}  ₹{self.amount} to {self.payee} ({mode_name(self.mode)})"


class Account:
    def __init__(self, owner, balance, pin):
        self.owner = owner
        self.__balance = balance
        self.__pin = pin
        self.history = []

    def get_balance(self):
        return self.__balance

    def check_pin(self, pin):
        return pin == self.__pin

    def fee_for(self, mode):
        if mode == "card":
            return 5
        elif mode == "netbanking":
            return 2
        return 0

    def cashback_for(self, amount):
        if amount >= 5000:
            return 100
        elif amount >= 2000:
            return 50
        elif amount >= 500:
            return 10
        return 0

    def add_money(self, amount):
        if amount <= 0:
            print("Invalid amount. Nothing added.")
            return
        self.__balance += amount
        save_balance(self.__balance)
        print(f"Added ₹{amount}. Balance ₹{self.__balance}")

    def pay(self, upi_id, amount, mode="upi"):
        if not is_valid_upi_id(upi_id):
            print(f"'{upi_id}' is not a valid UPI ID. Payment refused.")
            return
        if amount <= 0 or amount > 100000:
            print("Invalid amount. Payment refused.")
            return
        if mode_name(mode) == "Unknown mode":
            print(f"Unknown payment mode '{mode}'. Payment refused.")
            return
        fee = self.fee_for(mode)
        if amount + fee > self.__balance:
            print("Insufficient balance. Payment refused.")
            return
        cashback = self.cashback_for(amount)
        self.__balance = self.__balance - amount - fee + cashback
        payee = name_for(upi_id)
        t = Transaction(payee, amount, mode, fee, cashback, datetime.date.today().isoformat())
        self.history.append(t)
        save_payment(t)
        save_balance(self.__balance)
        print(f"Paid ₹{amount} to {payee} via {mode_name(mode)}")
        print(f"Fee ₹{fee} | Cashback ₹{cashback} | Balance ₹{self.__balance}")

    def statement(self):
        if len(self.history) == 0:
            print("No payments yet.")
            return
        amounts = list(map(lambda t: t.amount, self.history))
        total = reduce(lambda a, b: a + b, amounts)
        biggest = max(self.history, key=lambda t: t.amount)
        print(f"{self.owner}: {len(self.history)} payments | Spent ₹{total}")
        print(f"Biggest: {biggest.describe()}")
        for t in sorted(self.history, key=lambda t: t.amount):
            print("  " + t.describe())


class PremiumAccount(Account):
    def fee_for(self, mode):
        return 0                      # premium: no fees on any mode

    def cashback_for(self, amount):
        return super().cashback_for(amount) * 2


def ask_pin(account):
    for attempt in range(3):
        pin = input("UPI PIN: ").strip()
        if account.check_pin(pin):
            return True
        print(f"Wrong PIN. {2 - attempt} attempts left")
    return False


kind = clean(input("Account type (basic / premium): "))
balance = load_balance(1000)
if kind == "premium":
    account = PremiumAccount("Rajani", balance, "4321")
else:
    account = Account("Rajani", balance, "4321")
account.history = load_history()
print(f"Loaded {len(account.history)} saved payments. Balance ₹{account.get_balance()}")

choice = ""
while choice != "4":
    print()
    print("1 Pay  2 Balance  3 Statement  4 Exit  5 Add  6 Read SMS  7 Check PAN")
    choice = clean(input("Choose: "))

    if choice == "1":
        upi_id = clean(input("Pay to (UPI ID): "))
        amount = int(input("Amount in rupees: ").strip())
        mode = clean(input("Mode (upi / card / netbanking): "))
        if ask_pin(account):
            account.pay(upi_id, amount, mode)
        else:
            print("Three wrong PINs. Payment refused.")

    elif choice == "2":
        print(f"Balance: ₹{account.get_balance()}")

    elif choice == "3":
        account.statement()

    elif choice == "5":
        account.add_money(int(input("Amount to add: ").strip()))

    elif choice == "6":
        parsed = read_bank_sms(input("Paste the bank SMS: "))
        if parsed is None:
            print("Could not read that SMS.")
        else:
            amount, tail, payee = parsed
            print(f"₹{amount} to {payee} from the account ending {tail}")

    elif choice == "7":
        pan = clean(input("PAN: ")).upper()
        print("Valid PAN" if is_valid_pan(pan) else "Not a valid PAN")

    elif choice != "4":
        print(f"Unknown choice '{choice}'. Try again.")

print("Thank you for using PaisaPay")

# What v16 cannot do yet:
#   - stay correct if two copies run at the same time (both write the same files)
#   - keep the balance safe: anyone can open paisapay_balance.txt and type a new number
#   Real payment apps keep this data in a database on a server, for both reasons.


## Your turn

The assignment for this class: https://github.com/prateekScaler/AIML-Jul26-Batch3-Python/blob/main/Class-16-Exceptions-CSV-And-Modules/ASSIGNMENT.md

Add new cells below and write your own programs here.


In [ ]:
# your code
